# Genomic classification audit

Four classifiers, separate feature matrices and missing-data rules. Rebuild from selected variants, FISH measurements and declared panels embedded in the release AnnData, run native LymphGen/DLBclass/LymphPlex and an auditable HMRN adapter, compare with current calls, then replace the authoritative AnnData fields with a backup and verified atomic write.

**Input scope:** existing pathogenic-variant selection and AF >2% (or missing AF) are retained. This is not a new somatic-variant calling pipeline. Discovery panel assignments were recorded in the source workbook and are embedded with these inputs; validation BLYMFv2 remains a provisional study assumption. Absence of variants is not sufficient evidence that a patient was sequenced. Cases outside the explicit source manifest are listed. The explicitly declared discovery cases with unavailable source inputs have their unsupported calls cleared to unknown; other noneligible cases are retained.

**Method status:** HMRN observed-feature prediction is a research extension. LymphPlex runs the unmodified author function with numeric NA inputs, using native PAM missing-value handling; incomplete-panel performance was not separately validated in the publication. Coverage, exact-completion sensitivity and public example masking results are reported separately from subtype labels.

In [1]:
from pathlib import Path
import sys, json, shutil, importlib.util, inspect
import numpy as np
import pandas as pd
import h5py
from anndata.io import read_elem
ROOT = Path.cwd() if (Path.cwd()/'dlbcl').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT))
from dlbcl.classification_audit import (sha256, run, read_study, canonicalizer,
    preprocess_expression, promote_calls, CASE_KEYS)
from dlbcl.classification_audit import load_audit_config, require_audit_resources
config, CONFIG_PATH = load_audit_config(ROOT)
SOURCE = Path(config['source_adata']).resolve()
SOURCE_HASH = sha256(SOURCE)
tables, raw_expression = read_study(SOURCE)
canon = canonicalizer(config['hgnc'])
UPDATE_ADATA = config['promote']  # Explicit final-cell write switch; backup and atomic replacement.

OUT=Path(config['output_root'])/'genomic';OUT.mkdir(parents=True,exist_ok=True)
VENDOR=Path(config['vendor'])
require_audit_resources(config, 'genomic')
METHOD_FILES={str(p):sha256(p) for p in [CONFIG_PATH,ROOT/'dlbcl/classification_audit.py',ROOT/'dlbcl/genomic_classification_audit.py',ROOT/'tools/classification_audit/prepare_genomic.py',ROOT/'tools/classification_audit/run_lymphgen.R',ROOT/'tools/classification_audit/run_lymphplex.R',*sorted((ROOT/'config/dlbclass').glob('*')),*sorted((ROOT/'config/classification_audit').glob('hmrn*'))] if p.is_file()}


## 1. Rebuild classifier inputs from released selected variants, panel masks and measured FISH

The release contains selected mutation events and measurement availability in `uns["classification_inputs"]`, independently of the saved classifier calls. Raw variant calling and the upstream pathogenic/AF selection are outside this notebook. The preparation code is printed in full before execution. It retains approved gene/Entrez mappings, hotspot and truncating consequences, source variant coordinates, and per-case panel assignments. Measured-negative FISH is 0, positive is 1, unavailable is missing. No regional copy number is inferred. DLBclass encodes synonymous/nonsynonymous/SV events as 1/2/3 before feature-specific mean imputation. HMRN and LymphPlex have their own binary encodings in later cells.

In [2]:
prepare=ROOT/'tools/classification_audit/prepare_genomic.py'
print(prepare.read_text())
run([sys.executable,prepare],OUT/'prepare.log',cwd=ROOT)
variants=pd.read_csv(OUT/'selected_variants.tsv',sep='\t')
cases=pd.read_csv(OUT/'case_metadata.csv')
panel=pd.read_csv(OUT/'panel_coverage.tsv',sep='\t')
updates={};eligibility=[]
for short,cohort in [('discovery','Discovery'),('validation','Validation')]:
    ids=cases.loc[cases.cohort.eq(short),'patient_id']
    assert ids.is_unique
    canonical=pd.Index(tables[cohort].patient_id if 'patient_id' in tables[cohort] else tables[cohort].index)
    assert ids.isin(canonical).all()
    updates[cohort]=pd.DataFrame(index=ids)
    eligibility.extend(dict(cohort=cohort,patient_id=pid,eligible=pid in set(ids)) for pid in canonical)
pd.DataFrame(eligibility).to_csv(OUT/'eligibility.csv',index=False)
display(cases.groupby(['cohort','panel','panel_evidence']).size().rename('cases'))


"""Build fresh genomic classifier inputs from released selected events and availability."""
from pathlib import Path
import hashlib, json, sys
import h5py
import numpy as np
import pandas as pd
from anndata.io import read_elem
RELEASE=Path(__file__).resolve().parents[2]
sys.path.insert(0,str(RELEASE))
from dlbcl.classification_audit import load_audit_config
from dlbcl.lymphgen_portal import resolve_gene, fish_value, write_bundle
CONFIG,_=load_audit_config(RELEASE)
OUT=Path(CONFIG['output_root'])/'genomic'

def main():
    OUT.mkdir(parents=True,exist_ok=True)
    source=Path(CONFIG['source_adata'])
    with h5py.File(source,'r') as f:
        if 'uns/classification_inputs' not in f:
            raise ValueError('This audit requires the v1.1.0 AnnData containing uns["classification_inputs"]. Download the matching release bundle.')
        bundle=read_elem(f['uns/classification_inputs'])
    if bundle['schema']!='selected-genomic-inputs-v1':raise ValueError('Unsupported genomic-input sch

cohort      panel     panel_evidence                
discovery   BLYMF200  recorded in discovery workbook      4
            BLYMFv2   recorded in discovery workbook     57
validation  BLYMFv2   provisional BLYMFv2 assignment    221
Name: cases, dtype: int64

## 2. LymphGen 2.0: native reference model

Run separately for each panel using its complete measured-gene list, including genes with zero observed variants. Unknown FISH stays missing. No CN data means A53 is unavailable, not negative. MYD88 L265P and truncating mutations retain their explicit encoding. The native local model was previously verified against the official portal on identical inputs. [Wright et al.](https://pmc.ncbi.nlm.nih.gov/articles/PMC8459709/).

In [3]:
rscript=ROOT/'tools/classification_audit/run_lymphgen.R'
print(rscript.read_text())
for short,cohort in [('discovery','Discovery'),('validation','Validation')]:
    result=[]
    for panel_name in cases.loc[cases.cohort.eq(short),'panel'].unique():
        source=OUT/'inputs'/f'{short}_{panel_name}'
        dest=OUT/'LymphGen'/f'{short}_{panel_name}'
        run([config['rscript'],rscript,VENDOR/'LymphGen2',source,dest],dest/'run.log')
        d=pd.read_csv(dest/'Result.tsv',sep='\t').set_index('Sample.Name')
        expected=pd.read_csv(source/'sample_annotation.txt',sep='\t').Sample
        assert d.index.is_unique and set(d.index)==set(expected)
        result.append(d)
    d=pd.concat(result)
    assert set(d.index)==set(updates[cohort].index)
    updates[cohort]['Lymphgen']=d.loc[updates[cohort].index,'Subtype.Prediction']
    d.to_csv(OUT/f'{cohort}_lymphgen_scores.csv')
    display(updates[cohort].Lymphgen.value_counts().rename(cohort))


args <- commandArgs(trailingOnly=TRUE)
vendor <- args[1]; input <- args[2]; output <- args[3]
dir.create(output,recursive=TRUE,showWarnings=FALSE)
options(stringsAsFactors=FALSE)
source(file.path(vendor,'src/Model10_8.1.22.R'))
readcore <- function(name) read.table(file.path(vendor,'corefiles',name),header=TRUE,sep='\t')
samp <- read.table(file.path(input,'sample_annotation.txt'),header=TRUE,sep='\t')
flat <- read.table(file.path(input,'mutation_flat.txt'),header=TRUE,sep='\t')
genes <- read.table(file.path(input,'mutation_gene_list.txt'),header=TRUE,sep='\t')[[1]]
full <- as.matrix(readcore('Fullmat.2020.txt')==1)
study <- readcore('Study.samp.txt'); lab <- study[,18]; lab[lab=='SDT'] <- 'ST2'
feat <- readcore('Featord.2020.txt');feat[,11] <- feat[,11]==1
Index.Flat <- readcore('Index.Flat.txt')
result <- Predict9(flat,samp,Index.Flat,lab,
    Full.Uber.2020=readcore('Full.Uber.2020.txt'),Fullmat.2020=full,Featord.2020=feat,
    CGH.class=1,Has.Trunc=TRUE,mut.genelist=genes,CGH.geneli

Lymphgen
Other      32
MCD         9
EZB         9
BN2         7
EZB/ST2     2
MCD/ST2     1
ST2         1
Name: Discovery, dtype: int64

Lymphgen
Other    100
MCD       81
EZB       22
ST2       11
BN2        6
N1         1
Name: Validation, dtype: int64

## 3. DLBclass: author ensemble with reference-mean imputation

Unmeasured mutation, CN and rearrangement features receive their original encoded-feature means from the public 699-case reference. Mutation-negative is zero only for a measured gene. FISH positivity is a structural-variant input; merely sequencing MYC/BCL2/BCL6 does not measure a rearrangement. The native 163-to-21 reduction is evaluated with floating-point values so imputation is not truncated to integers. Execute all 500 author networks and their original confidence transformation. [DLBclass paper](https://doi.org/10.1182/blood.2024025652).

In [4]:
from dlbcl.dlbclass_targeted import run_official_ensemble
ref=pd.read_csv(ROOT/'config/dlbclass/feature_reference.csv',index_col=0)
weights=pd.read_csv(ROOT/'config/dlbclass/reduction_weights.csv',index_col=0)
for short,cohort in [('discovery','Discovery'),('validation','Validation')]:
    folder=OUT/'dlbclass'/short
    gsm=pd.read_csv(folder/'imputed_gsm.tsv',sep='\t',index_col=0)
    mask=pd.read_csv(folder/'measured_mask.tsv',sep='\t',index_col=0)
    assert not mask.loc[ref.type.eq('Regional CNV')].any().any()
    for pid in gsm:
        np.testing.assert_allclose(gsm.loc[~mask[pid],pid],ref.loc[~mask[pid],'reference_mean'])
    reduced=gsm.T @ weights.loc[gsm.index]
    scores,model_hash=run_official_ensemble(reduced,VENDOR/'DLBclass-tool')
    scores.to_csv(folder/'fresh_results.tsv',sep='\t')
    assert set(scores.index)==set(updates[cohort].index)
    updates[cohort]['DLBclass']=scores.loc[updates[cohort].index,'DLBclass']
    METHOD_FILES['DLBclass_500_networks']=model_hash
    display(scores.DLBclass.value_counts().rename(cohort))


loaded neural networks
Classifying 61 samples...


~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  net_inputs = torch.tensor(row, dtype=torch.float)
~/code/lymphotyper/vendor/DLBclass-tool/src/nn.py:21: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  x = F.softmax(x)
~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  net_inputs = torch.tensor(row, dtype=torch.float)
~/code/lymphotyper/vendor/DLBclass-tool/src/nn.py:21: UserWarning: Implicit dimension choice for softm

~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  net_inputs = torch.tensor(row, dtype=torch.float)
~/code/lymphotyper/vendor/DLBclass-tool/src/nn.py:21: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  x = F.softmax(x)
~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  net_inputs = torch.tensor(row, dtype=torch.float)
~/code/lymphotyper/vendor/DLBclass-tool/src/nn.py:21: UserWarning: Implicit dimension choice for softm

~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  net_inputs = torch.tensor(row, dtype=torch.float)
~/code/lymphotyper/vendor/DLBclass-tool/src/nn.py:21: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  x = F.softmax(x)
~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  net_inputs = torch.tensor(row, dtype=torch.float)
~/code/lymphotyper/vendor/DLBclass-tool/src/nn.py:21: UserWarning: Implicit dimension choice for softm

Done!


...

~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pred_df[idx] = average_output
~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  net_inputs = torch.tensor(row, dtype=torch.float)
~/code/lymphotyper/vendor/DLBclass-tool/src/nn.py:21: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  x = F.softmax(x)
~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:40: Performan

~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pred_df[idx] = average_output
~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  net_inputs = torch.tensor(row, dtype=torch.float)
~/code/lymphotyper/vendor/DLBclass-tool/src/nn.py:21: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  x = F.softmax(x)
~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:40: Performan

Done!


~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:40: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pred_df[idx] = average_output
~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:31: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  net_inputs = torch.tensor(row, dtype=torch.float)
~/code/lymphotyper/vendor/DLBclass-tool/src/nn.py:21: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  x = F.softmax(x)
~/code/lymphotyper/vendor/DLBclass-tool/src/classify_generic.py:40: Performan

DLBclass
C5    112
C2     55
C4     24
C3     20
C1     10
Name: Validation, dtype: int64

## 4. HMRN: restore the author reference fit and explicit missingness

The author reference fit is reproduced with RNG 3.5.3, seed 17, k=1…10, 1,000 starts and AIC. The six cluster sizes and enriched genes identify the published biological labels; we store names, not arbitrary component numbers. The fit and its complete-input predictions are frozen reference artifacts, independent of study labels. The executable reference-fit script is displayed below and can be rerun.

Corrected features include historical gene symbols, EZH2 Y641/Y646 isoforms, MYD88 L265P, and **mutation OR homozygous deletion**. A detected mutation proves the OR feature positive; no mutation with unmeasured CN does not prove it negative. The supplement describes broader SHM ascertainment for selected genes. A pathogenic-only export cannot establish those negatives: `_S` features retain observed mutations and otherwise remain missing.

For missing features, integrate them out of the Bernoulli likelihood (omit their terms). This is an explicit mathematical extension of the author's complete-input model, not its published validation. Export every feature's evidence and posterior; no zero filling. With no measured features, report Unassigned rather than classify from priors alone. [Author code](https://github.com/ecsg-uoy/DLBCLGenomicSubtyping/blob/master/fit_clusters.R), [supplement](https://api.repository.cam.ac.uk/server/api/core/bitstreams/8ecf06ef-3ab8-4356-8775-f6e7522cabcf/content).

In [5]:
from dlbcl.genomic_classification_audit import hmrn_features, bernoulli_posterior
print((ROOT/'tools/classification_audit/fit_hmrn.R').read_text())
print(inspect.getsource(hmrn_features)); print(inspect.getsource(bernoulli_posterior))
hmrn_meta=json.loads((ROOT/'config/classification_audit/hmrn_reference.json').read_text())
assert sha256(VENDOR/'DLBCLGenomicSubtyping/data/genomic_data.csv')==hmrn_meta['reference_data_sha256']
centers=pd.read_csv(ROOT/'config/classification_audit/hmrn_reference.centers.csv',index_col=0)
priors=pd.read_csv(ROOT/'config/classification_audit/hmrn_reference.priors.csv').set_index('component').prior.reindex(centers.columns)
reference=pd.read_csv(VENDOR/'DLBCLGenomicSubtyping/data/genomic_data.csv').set_index('PID')
ref_calls=pd.read_csv(ROOT/'config/classification_audit/hmrn_reference.calls.csv').set_index('PID').component
reference_posterior=bernoulli_posterior(reference[centers.index],centers,priors)
assert reference_posterior.idxmax(axis=1).eq(ref_calls).all()
native_posterior=pd.read_csv(ROOT/'config/classification_audit/hmrn_reference.posterior.csv').set_index('PID')
np.testing.assert_allclose(reference_posterior,native_posterior,atol=1e-12)
display(ref_calls.map(hmrn_meta['labels']).value_counts())
for short,cohort in [('discovery','Discovery'),('validation','Validation')]:
    c=cases[cases.cohort.eq(short)]
    v=variants[variants.cohort.eq(short)]
    fish=pd.read_csv(OUT/f'{short}_fish.csv',index_col=0)
    x,evidence=hmrn_features(v,c,panel,centers.index,fish,canon)
    posterior=bernoulli_posterior(x,centers,priors).rename(columns=hmrn_meta['labels'])
    calls=posterior.idxmax(axis=1).where(x.notna().any(axis=1),'Unassigned')
    updates[cohort]['HMRN']=calls.reindex(updates[cohort].index)
    posterior.to_csv(OUT/f'{cohort}_hmrn_posteriors.csv')
    x.to_csv(OUT/f'{cohort}_hmrn_features.csv');evidence.to_csv(OUT/f'{cohort}_hmrn_feature_evidence.csv',index=False)
    display(calls.value_counts().rename(cohort))


# Reproduce the author's reference fit; never fit on the query cohort.
args <- commandArgs(TRUE)
suppressPackageStartupMessages(library(flexmix))
reference <- read.csv(args[1], check.names=FALSE)
muts_df <- reference[,-1]
RNGversion('3.5.3'); set.seed(17)
ex <- initFlexmix(as.matrix(muts_df) ~ 1, k=1:10,
                 model=FLXMCmvbinary(), control=list(minprior=0.10), nrep=1000)
model <- getModel(ex, which='AIC')
saveRDS(list(aic=model, bic=getModel(ex,which='BIC'), fit=ex), args[2])
centers <- sapply(model@components, function(x) x[[1]]@parameters$center)
colnames(centers) <- paste0('reference_C',seq_len(ncol(centers)))
write.csv(centers, paste0(args[2],'.centers.csv'))
write.csv(data.frame(component=colnames(centers), prior=model@prior),paste0(args[2],'.priors.csv'),row.names=FALSE)
write.csv(data.frame(PID=reference$PID,component=paste0('reference_C',model@cluster)),paste0(args[2],'.calls.csv'),row.names=FALSE)
capture.output(sessionInfo(),file=paste0(args[2],'.session.txt'))

d

component
NEC           248
BCL2          176
MYD88         152
NOTCH2        143
SOCS1/SGK1    111
TET2/SGK1      98
Name: count, dtype: int64

MYD88         23
SOCS1/SGK1    16
NEC            6
TET2/SGK1      6
BCL2           6
NOTCH2         4
Name: Discovery, dtype: int64

MYD88         110
SOCS1/SGK1     44
NEC            32
NOTCH2         14
BCL2           11
TET2/SGK1      10
Name: Validation, dtype: int64

## 5. LymphPlex: native assignment with missing inputs

Run the unmodified author `GetCluster` function with numeric `NA` values. A unique observed seed pattern takes precedence; otherwise native PAM compares available features with the fixed centers. TP53 mutation overrides the result. The missing-distance rescaling is common across centers for each case. Measured negatives remain 0; off-panel genes and unavailable FISH remain missing. No zero or mean imputation is performed. TP53 must be measured, and non-TP53 cases need at least one measured center feature; this is an execution safeguard, not a validated minimum-coverage threshold.

The paper does not separately prescribe or validate incomplete-panel handling. Native PAM supports missing values. We retain feature coverage and exhaustive binary-completion sensitivity as annotations; disagreement among hypothetical completions does **not** prevent a native subtype call. `Others` is the native residual category. Cases with no genomic source data remain unknown outside this eligible cohort.

[Author code](https://github.com/difuSJTU/LymphPlex), [paper](https://www.nature.com/articles/s41392-023-01358-y), [PAM missing-value support](https://stat.ethz.ch/R-manual/R-devel/library/cluster/html/pam.html).

In [6]:
from dlbcl.genomic_classification_audit import (lymphplex_features, lymphplex_partial,
    lymphplex_observed, lymphplex_native)
print(inspect.getsource(lymphplex_native)); print(inspect.getsource(lymphplex_observed))
print((ROOT/'tools/classification_audit/run_lymphplex.R').read_text())
centers=pd.read_csv(VENDOR/'LymphPlex/Centers.txt',sep='\t',index_col=0)
seeds=pd.read_csv(VENDOR/'LymphPlex/Seeds.txt',sep='\t')
lymphplex_auxiliary={}
for short,cohort in [('discovery','Discovery'),('validation','Validation')]:
    c=cases[cases.cohort.eq(short)];v=variants[variants.cohort.eq(short)]
    fish=pd.read_csv(OUT/f'{short}_fish.csv',index_col=0)
    x=lymphplex_features(v,c,panel,centers,fish,canon)
    native=lymphplex_native(x,VENDOR/'LymphPlex',config['rscript'],OUT/f'{cohort}_lymphplex_native')
    sensitivity,completions=lymphplex_partial(x,centers,seeds)
    calls=sensitivity.rename(columns={'call':'completion_consensus','status':'completion_status'})
    calls.insert(0,'call',native)
    calls['completion_invariant']=calls.completion_consensus.ne('Unassigned')
    calls['n_measured']=x.notna().sum(axis=1);calls['n_required']=len(x.columns)
    calls['status']=np.where(x.isna().any(axis=1),'native_PAM_partial_inputs','native_complete_inputs')
    updates[cohort]['LymphPlex']=native.reindex(updates[cohort].index)
    x.to_csv(OUT/f'{cohort}_lymphplex_features.csv');calls.to_csv(OUT/f'{cohort}_lymphplex_audit.csv')
    completions.to_csv(OUT/f'{cohort}_lymphplex_completions.csv',index=False)
    prefix='uns' if cohort=='Discovery' else 'uns/validation_cohort'
    lymphplex_auxiliary[prefix+'/lymphplex_call_audit']=calls
    lymphplex_auxiliary[prefix+'/lymphplex_input_method']={
        'algorithm':'Author GetCluster with fixed PAM centers and TP53 override',
        'missing':'Numeric NA; native available-feature distances; no imputation',
        'sensitivity':'All binary completions are retained as a robustness annotation, not a call gate',
        'scope':'Cases with explicit genomic source inputs; TP53 measured',
        'limitation':'Incomplete-panel clinical accuracy not separately validated by authors',
        'publication':'https://www.nature.com/articles/s41392-023-01358-y',
        'author_code':'https://github.com/difuSJTU/LymphPlex'}
    display(calls.groupby(['call','completion_invariant']).size().rename(cohort))


def lymphplex_native(x, vendor, rscript, output):
    """Run author GetCluster/PAM on numeric NA inputs, with the TP53 override.

    The paper does not separately validate incomplete panels. Preserve feature
    coverage and completion sensitivity as annotations, not an abstention gate.
    """
    from .classification_audit import run
    vendor=Path(vendor);output=Path(output);output.mkdir(parents=True,exist_ok=True)
    centers=pd.read_csv(vendor/'Centers.txt',sep='\t',index_col=0)
    seeds=pd.read_csv(vendor/'Seeds.txt',sep='\t')
    columns=[*centers.columns,'TP53']
    if not x.index.is_unique or not set(columns)<=set(x): raise ValueError('Require unique cases and explicit classifier feature columns')
    inputs=x.loc[:,columns].astype(float).copy()
    expected={str(pid):lymphplex_observed(row,centers,seeds) for pid,row in inputs.iterrows()}
    inputs.index=inputs.index.astype(str)
    if not inputs.index.is_unique: raise ValueError('Case identifiers collide after string conv

call    completion_invariant
BN2     True                     3
EZB     False                    3
        True                     6
MCD     False                    3
        True                    13
N1      True                     1
Others  False                   10
        True                    10
ST2     False                    1
        True                     1
TP53    True                    10
Name: Discovery, dtype: int64

call    completion_invariant
BN2     False                    6
        True                     1
EZB     False                   17
        True                     2
MCD     False                   12
        True                    85
N1      True                     2
Others  False                   43
        True                    28
ST2     False                    6
        True                     4
TP53    True                    15
Name: Validation, dtype: int64

### Public example masking check

Rerun the author-supplied example matrix with full inputs and then mask the two BLYMFv2-missing genes, optionally also BCL6 or both BCL2/BCL6 rearrangements. Report agreement and per-label recall/precision against the full-input **native calls**, not against the unrelated label column in the example files. This is a reproducibility sensitivity check, not an independent clinical validation. TP53 priority contributes substantially to overall agreement; also show non-TP53 agreement and label-specific results. These external examples never define study labels or fit a new model.

In [7]:
example=pd.read_csv(VENDOR/'LymphPlex/Matrix.txt',sep='\t',dtype={'Sample':str}).set_index('Sample')
tp53=pd.read_csv(VENDOR/'LymphPlex/TP53.txt',sep='\t',dtype={'Sample':str}).set_index('Sample')
assert example.index.equals(tp53.index)
example['TP53']=tp53.TP53
baseline=lymphplex_native(example,VENDOR/'LymphPlex',config['rscript'],OUT/'lymphplex_example/full')
masks={'genes':['DDX3X','ZFP36L1'],
       'genes_BCL6':['DDX3X','ZFP36L1','BCL6'],
       'genes_both_FISH':['DDX3X','ZFP36L1','BCL2','BCL6']}
summary=[];metrics=[]
for name,features in masks.items():
    masked=example.copy();masked[features]=np.nan
    calls=lymphplex_native(masked,VENDOR/'LymphPlex',config['rscript'],OUT/'lymphplex_example'/name)
    matches=calls.eq(baseline);non_tp53=baseline.ne('TP53')
    summary.append(dict(mask=name,n=len(calls),matches=int(matches.sum()),agreement=float(matches.mean()),
                        non_TP53_n=int(non_tp53.sum()),non_TP53_agreement=float(matches[non_tp53].mean())))
    pd.crosstab(baseline,calls,rownames=['full'],colnames=['masked']).to_csv(OUT/f'lymphplex_example_{name}_confusion.csv')
    for label in sorted(set(baseline)|set(calls)):
        true=baseline.eq(label);pred=calls.eq(label);tp=int((true&pred).sum())
        metrics.append(dict(mask=name,label=label,n_full=int(true.sum()),n_masked=int(pred.sum()),
            recall=tp/true.sum() if true.any() else np.nan,precision=tp/pred.sum() if pred.any() else np.nan))
summary=pd.DataFrame(summary);metrics=pd.DataFrame(metrics)
summary.to_csv(OUT/'lymphplex_example_agreement.csv',index=False)
metrics.to_csv(OUT/'lymphplex_example_label_metrics.csv',index=False)
display(summary);display(metrics)


,mask,n,matches,agreement,non_TP53_n,non_TP53_agreement
0,genes,616,606,0.983766,314,0.968153
1,genes_BCL6,616,578,0.938312,314,0.878981
2,genes_both_FISH,616,563,0.913961,314,0.831210


,mask,label,n_full,n_masked,recall,precision
0,genes,BN2,45,45,1.000000,1.000000
1,genes,EZB,21,24,1.000000,0.875000
2,genes,MCD,68,67,0.985294,1.000000
3,genes,N1,14,14,1.000000,1.000000
4,genes,Others,139,136,0.964029,0.985294
5,genes,ST2,27,28,0.851852,0.821429
6,genes,TP53,302,302,1.000000,1.000000
7,genes_BCL6,BN2,45,36,0.600000,0.750000
8,genes_BCL6,EZB,21,24,0.904762,0.791667
9,genes_BCL6,MCD,68,68,0.955882,0.955882


## 6. Before/after audit and authoritative replacement

Promotion includes only the explicitly eligible genomic cases. Cases with unavailable source inputs are listed in `eligibility.csv`. T29, T41 and T48 have their unsupported calls cleared to unknown, as explicitly specified in the configuration; their old calls remain in the audit backups. Other noneligible cases are retained. Absence from this source is not converted into a negative genotype. The current source hash is checked again and unrelated AnnData contents are verified unchanged. HMRN biological names and native LymphPlex labels are kept separate from missing-input and sensitivity annotations. Unavailable genomic source inputs remain unknown.

In [8]:
# Explicit source-unavailability correction; never infer a negative genotype from no rows.
cleared=[]
for pid in config.get('clear_unavailable_discovery_calls',[]):
    if pid not in updates['Discovery'].index:
        assert pid in set(tables['Discovery'].patient_id)
        updates['Discovery'].loc[pid,['Lymphgen','DLBclass','HMRN','LymphPlex']]='unknown'
        cleared.append(pid)
for cohort,new in updates.items():
    new['genomic_classification_status']='classified from available source inputs'
updates['Discovery'].loc[cleared,'genomic_classification_status']='source inputs unavailable'
rows=[]
for cohort,new in updates.items():
    old=tables[cohort].copy()
    if 'patient_id' in old: old=old.set_index('patient_id',drop=False)
    assert not new.isna().any().any()
    new.to_csv(OUT/f'{cohort}_fresh_calls.csv')
    for col in new:
        rows.append(dict(cohort=cohort,classifier=col,n=len(new),changed=int((old[col].reindex(new.index).astype(str) if col in old else pd.Series('',index=new.index)).ne(new[col].astype(str)).sum())))
display(pd.DataFrame(rows))
# Replace accompanying DLBclass scores/method metadata so they cannot describe old calls.
auxiliary=dict(lymphplex_auxiliary)
for short,cohort in [('discovery','Discovery'),('validation','Validation')]:
    prefix='uns' if cohort=='Discovery' else 'uns/validation_cohort'
    scores=pd.read_csv(OUT/'dlbclass'/short/'fresh_results.tsv',sep='\t',index_col=0)
    auxiliary[prefix+'/dlbclass_mean_scores']=scores[['C1','C2','C3','C4','C5','Confidence']]
    auxiliary[prefix+'/dlbclass_input_method']={
        'date':'2026-10-17','missing':'699-reference encoded-feature means',
        'coverage':'Discovery: recorded BLYMF200/BLYMFv2; validation: provisional BLYMFv2',
        'eligibility':'Explicit genomic source manifest; scores include only freshly classified cases',
        'fish':'Missing imputed, negative zero, positive three','regional_cnv':'Unmeasured and imputed',
        'mutation_encoding':'Selected synonymous=1, nonsynonymous=2',
        'reduction':'Author linear reduction, floating-point means preserved'}
auxiliary['uns/lymphgen_discovery_method']={
    'date':'2026-10-17','source':'Native LymphGen 2.0 rerun from panel-specific mutation/FISH inputs',
    'copy_number':'Unavailable; A53 not evaluated','coverage':'Recorded BLYMF200/BLYMFv2',
    'case_count':str(int(cases.cohort.eq('discovery').sum()))}

provenance={'config':config,'method_files':METHOD_FILES,'input_manifest':json.loads((OUT/'input_manifest.json').read_text()),
            'hmrn_reference':hmrn_meta,'cleared_unavailable_discovery_calls':cleared,'HMRN_partial_input':'Observed-feature Bernoulli likelihood; CN and incomplete SHM negatives marginalized',
            'LymphPlex_partial_input':'Native author GetCluster/PAM with numeric NA, observed-feature distances, TP53 override; no imputation; completion sensitivity does not gate labels',
            'limitations':'Pathogenic-selected variants; panel coverage is not sample-level callable-base coverage; validation BLYMFv2 provisional'}
(OUT/'run_provenance.json').write_text(json.dumps(provenance,indent=2))
if UPDATE_ADATA:
    changes=promote_calls(SOURCE,updates,OUT/'promotion',audit_name='genomic',run_provenance=provenance,expected_source_sha256=SOURCE_HASH,auxiliary_updates=auxiliary)
    display(changes.groupby(['cohort','column']).changed.agg(['size','sum']))
else: print('Review-only run; UPDATE_ADATA is False.')


,cohort,classifier,n,changed
0,Discovery,Lymphgen,64,0
1,Discovery,DLBclass,64,0
2,Discovery,HMRN,64,0
3,Discovery,LymphPlex,64,0
4,Discovery,genomic_classification_status,64,0
5,Validation,Lymphgen,221,0
6,Validation,DLBclass,221,0
7,Validation,HMRN,221,0
8,Validation,LymphPlex,221,0
9,Validation,genomic_classification_status,221,0


Review-only run; UPDATE_ADATA is False.
